# BIOMASS tomogram provider

This notebook inspects both TOM L1D and L2A products without loading their multi-gigabyte measurement arrays, then opens a small L2A HV subset for visualization. Install the optional dependencies with `python -m pip install -e '.[tomogram,examples]'`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from tomo_canopy.providers import TomogramProvider
from tomo_canopy.types import DataLayer, GeometryType, Modality

## Inspect both products

`inspect()` reads only STAC and XML metadata. It is safe to use for inventory and product selection.

In [ ]:
data_candidates = [Path("data/TOM"), Path("../data/TOM")]
TOM_DIRECTORY = next(
    (path for path in data_candidates if path.is_dir()), data_candidates[0]
)
provider = TomogramProvider()
products = [provider.inspect(path) for path in sorted(TOM_DIRECTORY.iterdir())]
for product in products:
    print(
        product.product_level, product.shape, product.polarizations,
        (product.vertical_minimum, product.vertical_maximum), product.bbox,
    )

## Open L1D in native SAR coordinates

L1D is not yet a geographic volume. Its axes remain relative azimuth time, slant-range time, relative height, and polarization.

In [ ]:
l1d_product = next(product for product in products if product.product_level == "L1D")
l1d = provider.open(
    l1d_product.root, polarization="HV", height_range=(-20.0, 60.0),
    chunks="auto",
)
l1d

## Open a bounded L2A geographic volume

The subset is centred on the product footprint and spans about 0.04° in each horizontal direction. Data is read only when the plots request values.

In [ ]:
l2a_product = next(product for product in products if product.product_level == "L2A")
# Derive the centre from normalized coordinates. Some prototype products
# store their raw STAC bbox latitude-first, so product.bbox is not used here.
footprint = provider.open(
    l2a_product.root, polarization="HV", height_range=(0.0, 0.0),
    chunks="auto",
)
centre_x = float(footprint.x.mean())
centre_y = float(footprint.y.mean())
footprint.close()
l2a = provider.open(
    l2a_product.root,
    polarization="HV",
    height_range=(-20.0, 60.0),
    bounds=(centre_x - 0.02, centre_y - 0.02, centre_x + 0.02, centre_y + 0.02),
    chunks="auto",
)
l2a

In [ ]:
horizontal_slice = l2a.tomographic_cube.sel(
    polarization="HV", z=10.0, method="nearest"
).compute()
horizontal_slice.plot(
    
    x="x", y="y", robust=True, figsize=(8, 6),
)
plt.title("L2A HV horizontal slice")
plt.show()

In [ ]:
vertical_slice = l2a.tomographic_cube.sel(
    polarization="HV", y=centre_y, method="nearest"
).compute()
vertical_slice.plot(x="x", y="z", robust=True, figsize=(10, 5))
plt.title("L2A HV vertical slice")
plt.show()

## Wrap the geographic volume for the shared interface

The provider preserves the relative-height semantics in attributes. Reprojection and conversion to an absolute elevation grid belong in the tomogram processor.

In [ ]:
tomogram_layer = DataLayer(
    name="biomass_tomogram", data=l2a,
    modality=Modality.TOMOGRAM, geometry=GeometryType.VOLUME,
)
tomogram_layer